## Crash severity

The fixed rules are in `src/data/severity.py`. See [severity rules](../data/README.md#crash-severity-rules)
for precedence, missing outcomes, and the training class counts.

We review labels using 2020-2022 only, then apply the same rules to every year.
`unknown` stays in the saved files. Exclude it from model training and evaluation
and report how many rows were removed.

Outcome fields and label reasons must not be predictors. IDs and survey-design
fields are not predictors either; use `WEIGHT` only for sample weighting.

The classifier also returns `INJURY_ONLY_SEVERITY` for the training comparison.
Both labels are outcomes, never predictors. We predict severity categories using
classification models. Review [predictor availability](../data/README.md#predictor-availability)
before claiming prediction before the outcome is known.


In [ ]:
import pandas as pd
from pathlib import Path

years = [2020, 2021, 2022, 2023, 2024]
# Run with notebooks/ as the working directory.
NOTEBOOK_DIR = Path.cwd().resolve()
PROJECT_ROOT = NOTEBOOK_DIR.parent
if NOTEBOOK_DIR.name != "notebooks" or not (PROJECT_ROOT / "data" / "README.md").is_file():
    raise RuntimeError("Run this notebook with the repository's notebooks/ directory as the working directory.")
# Import the shared checks from the repository.
import sys
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))
from src.data.joins import audit_raw_joins, merge_crash_summary
from src.data.load import read_raw_csv
from src.data.audit_codes import validate_codes
from src.data.severity import classify_crashes, training_distribution, write_training_reports

RAW = PROJECT_ROOT / "data" / "raw"
#Claude help me build the for loop to save the headache of data upload. 
def load_table(name):
    frames = []
    for y in years:
        path = RAW / str(y) / f"{name}.csv"
        df = read_raw_csv(path)
        checks = validate_codes(df, name, y)
        if any(check["rows"] for check in checks):
            raise ValueError(f"{y} {name}: values outside the reviewed code sets")
        df["YEAR_FILE"] = y
        frames.append(df)
    return pd.concat(frames, ignore_index=True)

In [ ]:
#Creates three tables for the three seperate CSV shapes. 
accident = load_table("accident")
vehicle = load_table("vehicle")
person = load_table("person")

# Check raw keys and coverage before grouping or joining.
join_audit = audit_raw_joins(accident, vehicle, person)
join_audit


In [ ]:
# Apply the fixed rules. This does not fit thresholds or choose rules from the data.
labels = classify_crashes(accident, vehicle, person)

# Review class counts and reasons using training years only.
training_labels = labels.loc[labels["YEAR_FILE"].isin([2020, 2021, 2022])]
severity_report = training_distribution(training_labels)
write_training_reports(training_labels, PROJECT_ROOT / "docs")
severity_report


In [ ]:
# Keep every accident once when attaching the crash-level severity summary.
accident = merge_crash_summary(accident, labels[["YEAR_FILE", "CASENUM", "SEVERITY"]])
if accident["SEVERITY"].isna().any():
    raise ValueError("Every crash must have a severity label")

# Record the final count for each year alongside raw-table coverage.
joined_counts = accident.groupby("YEAR_FILE").size()
join_audit["joined_crash_rows"] = join_audit["year"].map(joined_counts)
if not join_audit["joined_crash_rows"].equals(join_audit["accident_rows"]):
    raise ValueError("Crash counts changed after joining")
AUDIT_PATH = PROJECT_ROOT / "docs" / "crss_join_audit.csv"
AUDIT_PATH.parent.mkdir(parents=True, exist_ok=True)
join_audit.to_csv(AUDIT_PATH, index=False)
print(accident.shape)
join_audit


In [13]:
#Remove columns that will cause data leakage by giving away the result of the crash or any other relevant information. 
leak_cols = ["MAX_SEV", "MAX_SEVNAME", "NUM_INJ", "NUM_INJNAME", "MAXSEV_IM", "MAXSEV_IMNAME", "NO_INJ_IM", "NO_INJ_IMNAME", "STRATUM", "STRATUMNAME", "PSUSTRAT"]
accident = accident.drop(columns=leak_cols)

In [14]:
#Seperate the data into our three splits. Train 2020-2022, Validation 2023, Test 2024.
train = accident[accident["YEAR_FILE"].isin([2020, 2021, 2022])].copy()
validation = accident[accident["YEAR_FILE"] == 2023].copy()
test = accident[accident["YEAR_FILE"] == 2024].copy()
#Looking at shape, ensuring that all three have the same amount of columns. 
print(train.shape, validation.shape, test.shape)

(162900, 71) (50103, 71) (51658, 71)


## Missing values and codes

The shared reader preserves valid text such as `WRK_ZONENAME = "None"`.
Only empty CSV cells are read as null. We do not fill work-zone labels or replace
8/9/98/99 globally: those numbers mean different things in different fields.

See [missing values and coding consistency](../data/README.md#missing-values-and-coding-consistency)
for the reports and handling rules. Regenerate them from the repository root
with `python -m src.data.audit_codes`. Learn imputation and encoding from training
only; validation and test checks must not change those choices.


In [23]:
#Save processed data
#Claude helped me write this as I couldn't get the path to work as described by the GitHub Repo. 
OUT = PROJECT_ROOT / "data" / "processed"
OUT.mkdir(parents=True, exist_ok=True)
train.to_csv(OUT / "train.csv", index=False)
validation.to_csv(OUT / "validation.csv", index=False)
test.to_csv(OUT / "test.csv", index=False)

## Output and Handoff

### Files saved
Saved to `data/processed/` (local only, not committed to GitHub):

| File | Years | Rows | Purpose |
|---|---|---|---|
| `train.csv` | 2020-2022 | 162,900 | Model training and fitting preprocessing |
| `validation.csv` | 2023 | 50,103 | Hyperparameter tuning and model selection |
| `test.csv` | 2024 | 51,658 | Final evaluation only |

Each file has 71 columns, one row per crash, with the target in the `SEVERITY` column (`high`, `moderate`, `low`, `unknown`).

### Class counts
See [the training report](../docs/crss_severity_training.csv) for 2020-2022
counts and label reasons. Holdout outcomes are not used to choose label rules.

### Still to do (not done in this notebook)
- Filter `unknown` rows before training and evaluation, and report how many were removed.
- Choose features (core: roadway, environment, vehicle; expanded: plus driver-related). The `*NAME` columns are text copies of codes and can be dropped.
- Treat code columns as categories and handle unknown codes (8, 9, 98, 99).
- Fit all imputing, encoding and scaling on `train` only.
- Do not use `test.csv` for any decisions. Use `WEIGHT` only as a sample weight.